# Listing Mortgage Rate Enrichment 

In [4]:
import pandas as pd

url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=MORTGAGE30US"
mortgage = pd.read_csv(url)

# Rename columns
mortgage.columns = ['date', 'rate_30yr_fixed']

# Convert to datetime
mortgage['date'] = pd.to_datetime(mortgage['date'])
mortgage.head()

,date,rate_30yr_fixed
0,1971-04-02,7.33
1,1971-04-09,7.31
2,1971-04-16,7.31
3,1971-04-23,7.31
4,1971-04-30,7.29


In [5]:
mortgage['year_month'] = mortgage['date'].dt.to_period('M')
mortgage_monthly = (
    mortgage.groupby('year_month')['rate_30yr_fixed']
    .mean()
    .reset_index()
)

mortgage_monthly.head()

,year_month,rate_30yr_fixed
0,1971-04,7.3100
1,1971-05,7.4250
2,1971-06,7.5300
3,1971-07,7.6040
4,1971-08,7.6975


In [7]:
listings = pd.read_csv("../../IDX_Data/listing_filtered_week2_3.csv")
listings['year_month'] = pd.to_datetime(listings['ListingContractDate']).dt.to_period('M')
listing_with_rates = listings.merge(mortgage_monthly, on='year_month', how='left')

print("Missing mortgage rates:", listing_with_rates['rate_30yr_fixed'].isnull().sum())

/var/folders/hx/s9px6scd2pl5dqfp0pldbdgc0000gn/T/ipykernel_21329/221604871.py:1: DtypeWarning: Columns (0: ListAgentEmail, 1: BuyerAgencyCompensationType) have mixed types. Specify dtype option on import or set low_memory=False.
  listings = pd.read_csv("../../IDX_Data/listing_filtered_week2_3.csv")


Missing mortgage rates: 0


In [8]:
listing_with_rates[['ListingContractDate', 'year_month', 'ListPrice', 'rate_30yr_fixed']].head()

,ListingContractDate,year_month,ListPrice,rate_30yr_fixed
0,2024-01-01,2024-01,1340000.0,6.6425
1,2024-01-24,2024-01,2500000.0,6.6425
2,2024-01-12,2024-01,3150000.0,6.6425
3,2024-01-20,2024-01,3090000.0,6.6425
4,2024-01-12,2024-01,12725000.0,6.6425


In [9]:
listing_with_rates['rate_30yr_fixed'].describe()

count    540624.000000
mean          6.616429
std           0.302798
min           6.047500
25%           6.352500
50%           6.720000
75%           6.842500
max           7.060000
Name: rate_30yr_fixed, dtype: float64

In [10]:
listing_with_rates.to_csv("listing_with_rates.csv", index=False)
print("Listing with mortgage rates saved.")

Listing with mortgage rates saved.


### Merge Validation

Mortgage rate data was successfully merged into the listing dataset using a monthly key. 

No missing values were found in the mortgage rate column, confirming a complete and accurate merge.